In [231]:
import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, PolynomialFeatures
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, confusion_matrix

import category_encoders as ce      # 'category_encoders' is used for Binary Encoding

import warnings
warnings.filterwarnings('ignore')

In [232]:
df = pd.read_csv('house price prediction.csv')
print(df.columns)
df.head()

Index(['date', 'price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot',
       'floors', 'waterfront', 'view', 'condition', 'sqft_above',
       'sqft_basement', 'yr_built', 'yr_renovated', 'street', 'city',
       'statezip', 'country'],
      dtype='object')


,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,street,city,statezip,country
0,2014-05-02 00:00:00,313000.0,3.0,1.50,1340,7912,1.5,0,0,3,1340,0,1955,2005,18810 Densmore Ave N,Shoreline,WA 98133,USA
1,2014-05-02 00:00:00,2384000.0,5.0,2.50,3650,9050,2.0,0,4,5,3370,280,1921,0,709 W Blaine St,Seattle,WA 98119,USA
2,2014-05-02 00:00:00,342000.0,3.0,2.00,1930,11947,1.0,0,0,4,1930,0,1966,0,26206-26214 143rd Ave SE,Kent,WA 98042,USA
3,2014-05-02 00:00:00,420000.0,3.0,2.25,2000,8030,1.0,0,0,4,1000,1000,1963,0,857 170th Pl NE,Bellevue,WA 98008,USA
4,2014-05-02 00:00:00,550000.0,4.0,2.50,1940,10500,1.0,0,0,4,1140,800,1976,1992,9105 170th Ave NE,Redmond,WA 98052,USA


In [233]:
# changing column order
df = df[['date', 'country', 'statezip', 'city', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated',  'price']]

# printing unique values
print(df.nunique())

print('\nSHAPE OF THE DATA : ', df.shape)
df.head()


date               70
country             1
statezip           77
city               44
bedrooms           10
bathrooms          26
sqft_living       566
sqft_lot         3113
floors              6
waterfront          2
view                5
condition           5
sqft_above        511
sqft_basement     207
yr_built          115
yr_renovated       60
price            1741
dtype: int64

SHAPE OF THE DATA :  (4600, 17)


,date,country,statezip,city,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,price
0,2014-05-02 00:00:00,USA,WA 98133,Shoreline,3.0,1.50,1340,7912,1.5,0,0,3,1340,0,1955,2005,313000.0
1,2014-05-02 00:00:00,USA,WA 98119,Seattle,5.0,2.50,3650,9050,2.0,0,4,5,3370,280,1921,0,2384000.0
2,2014-05-02 00:00:00,USA,WA 98042,Kent,3.0,2.00,1930,11947,1.0,0,0,4,1930,0,1966,0,342000.0
3,2014-05-02 00:00:00,USA,WA 98008,Bellevue,3.0,2.25,2000,8030,1.0,0,0,4,1000,1000,1963,0,420000.0
4,2014-05-02 00:00:00,USA,WA 98052,Redmond,4.0,2.50,1940,10500,1.0,0,0,4,1140,800,1976,1992,550000.0


### **Approach for Feature Selection**


- OneHotEncoder - country
- For city we will be using Binary Encoder, applying One hot encoder on it will be messy
- Remove the 'WA' in statezip
- Covert date into date format



In [234]:
# applying one hot encoding on country

ohe = OneHotEncoder(sparse_output=False)
country = ohe.fit_transform(df[['country']])

df['country'] = pd.DataFrame(country)

df.head()



,date,country,statezip,city,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,price
0,2014-05-02 00:00:00,1.0,WA 98133,Shoreline,3.0,1.50,1340,7912,1.5,0,0,3,1340,0,1955,2005,313000.0
1,2014-05-02 00:00:00,1.0,WA 98119,Seattle,5.0,2.50,3650,9050,2.0,0,4,5,3370,280,1921,0,2384000.0
2,2014-05-02 00:00:00,1.0,WA 98042,Kent,3.0,2.00,1930,11947,1.0,0,0,4,1930,0,1966,0,342000.0
3,2014-05-02 00:00:00,1.0,WA 98008,Bellevue,3.0,2.25,2000,8030,1.0,0,0,4,1000,1000,1963,0,420000.0
4,2014-05-02 00:00:00,1.0,WA 98052,Redmond,4.0,2.50,1940,10500,1.0,0,0,4,1140,800,1976,1992,550000.0


In [235]:
# applying Binary Encoder on 'city'
encoder = ce.BinaryEncoder(cols='city')        # make sure, here you must mention the feature name for which you want to use Binary Encoder

df = encoder.fit_transform(df)                 # keep in mind, BinaryEncoder only expects a pandas DataFrame, you cant put any list or array. Else will throw error
df.head()

,date,country,statezip,city_0,city_1,city_2,city_3,city_4,city_5,bedrooms,...,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,price
0,2014-05-02 00:00:00,1.0,WA 98133,0,0,0,0,0,1,3.0,...,7912,1.5,0,0,3,1340,0,1955,2005,313000.0
1,2014-05-02 00:00:00,1.0,WA 98119,0,0,0,0,1,0,5.0,...,9050,2.0,0,4,5,3370,280,1921,0,2384000.0
2,2014-05-02 00:00:00,1.0,WA 98042,0,0,0,0,1,1,3.0,...,11947,1.0,0,0,4,1930,0,1966,0,342000.0
3,2014-05-02 00:00:00,1.0,WA 98008,0,0,0,1,0,0,3.0,...,8030,1.0,0,0,4,1000,1000,1963,0,420000.0
4,2014-05-02 00:00:00,1.0,WA 98052,0,0,0,1,0,1,4.0,...,10500,1.0,0,0,4,1140,800,1976,1992,550000.0


In [236]:
# removing 'WA' from all the values in 'statezip'

# df['statezip'] = df['statezip'].str.split().str[-1].astype(int)
df['statezip'] = df['statezip'].str.replace('WA', '')
df['statezip'] = df['statezip'].astype('int')           # convering numbers into integers

# print(df.info())
df.head()


,date,country,statezip,city_0,city_1,city_2,city_3,city_4,city_5,bedrooms,...,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,price
0,2014-05-02 00:00:00,1.0,98133,0,0,0,0,0,1,3.0,...,7912,1.5,0,0,3,1340,0,1955,2005,313000.0
1,2014-05-02 00:00:00,1.0,98119,0,0,0,0,1,0,5.0,...,9050,2.0,0,4,5,3370,280,1921,0,2384000.0
2,2014-05-02 00:00:00,1.0,98042,0,0,0,0,1,1,3.0,...,11947,1.0,0,0,4,1930,0,1966,0,342000.0
3,2014-05-02 00:00:00,1.0,98008,0,0,0,1,0,0,3.0,...,8030,1.0,0,0,4,1000,1000,1963,0,420000.0
4,2014-05-02 00:00:00,1.0,98052,0,0,0,1,0,1,4.0,...,10500,1.0,0,0,4,1140,800,1976,1992,550000.0


In [237]:
# applying datetime format on 'date'

df['date'] = pd.to_datetime(df['date'])     
# keep in mind, after converting it into datetime format, if you run this feature in linear reg, it will throw error
# hence you have to drop this feature. But if you drop this feature you would loose all the seasonal trends that how it affects the price
# to avoid this, you have to extract new features for day, month, and year


# day
df['day'] = df['date'].dt.dayofweek

# month
df['month'] = df['date'].dt.month

# year
df['year'] = df['date'].dt.year



df.head()

,date,country,statezip,city_0,city_1,city_2,city_3,city_4,city_5,bedrooms,...,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,price,day,month,year
0,2014-05-02,1.0,98133,0,0,0,0,0,1,3.0,...,0,3,1340,0,1955,2005,313000.0,4,5,2014
1,2014-05-02,1.0,98119,0,0,0,0,1,0,5.0,...,4,5,3370,280,1921,0,2384000.0,4,5,2014
2,2014-05-02,1.0,98042,0,0,0,0,1,1,3.0,...,0,4,1930,0,1966,0,342000.0,4,5,2014
3,2014-05-02,1.0,98008,0,0,0,1,0,0,3.0,...,0,4,1000,1000,1963,0,420000.0,4,5,2014
4,2014-05-02,1.0,98052,0,0,0,1,0,1,4.0,...,0,4,1140,800,1976,1992,550000.0,4,5,2014


In [238]:
df2 = df[['year', 'month', 'day', 'country', 'statezip', 'city_0', 'city_1', 'city_2', 'city_3', 'city_4', 'city_5', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'price']]

df2.head()

,year,month,day,country,statezip,city_0,city_1,city_2,city_3,city_4,...,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,price
0,2014,5,4,1.0,98133,0,0,0,0,0,...,7912,1.5,0,0,3,1340,0,1955,2005,313000.0
1,2014,5,4,1.0,98119,0,0,0,0,1,...,9050,2.0,0,4,5,3370,280,1921,0,2384000.0
2,2014,5,4,1.0,98042,0,0,0,0,1,...,11947,1.0,0,0,4,1930,0,1966,0,342000.0
3,2014,5,4,1.0,98008,0,0,0,1,0,...,8030,1.0,0,0,4,1000,1000,1963,0,420000.0
4,2014,5,4,1.0,98052,0,0,0,1,0,...,10500,1.0,0,0,4,1140,800,1976,1992,550000.0


In [241]:
X = df2.drop(columns=['price'])
y = df2['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred = pd.DataFrame(lr.predict(X_test))

# y_pred


print(f"ACCURACY SCORE : {accuracy_score(y_test, y_pred)}")




ValueError: continuous is not supported